# Bronze to silver - Cleaning & transforming fact table

In [0]:
from pyspark.sql.types import IntegerType, FloatType
import pyspark.sql.functions as F
catalog_name = 'edutrack'

In [0]:
df = spark.table('edutrack.bronze.brz_enrollments')
print('Row count:', df.count())
df.printSchema()

### Step 1 - Removing duplicates on enrollment_id

In [0]:
print('Duplicates:', df.groupBy('enrollment_id').count().filter(F.col('count')>1).count())
df = df.dropDuplicates(['enrollment_id'])
print('After dedup:', df.count())

### Step 2 - Fixing marks_obtained (remove %, cast to int, floor negatives to 0)

In [0]:
df.select('marks_obtained').show(10, truncate=False)

In [0]:
df = df.withColumn('marks_obtained', F.regexp_replace(F.col('marks_obtained'),'[%]','').cast(IntegerType()))
df = df.withColumn('marks_obtained', F.when(F.col('marks_obtained')<0,0).otherwise(F.col('marks_obtained')))
print('Negative marks fixed. Sample:')
df.select('marks_obtained').show(5)

### Step 3 - Caping attendance_pct at 100

In [0]:
print('Attendance > 100:', df.filter(F.col('attendance_pct').cast(IntegerType())>100).count())
df = df.withColumn('attendance_pct',
    F.when(F.col('attendance_pct').cast(IntegerType())>100, 100)
     .otherwise(F.col('attendance_pct').cast(IntegerType())))

### Step 4 - Standardising grade values

In [0]:
df.select('grade').distinct().orderBy('grade').show()

In [0]:
df = df.withColumn('grade',
    F.when(F.upper(F.col('grade')).isin('A','A+'), 'A')
     .when(F.upper(F.col('grade')).isin('B','B+'), 'B')
     .when(F.upper(F.col('grade')).isin('C','PASS','Pass','pass'), 'C')
     .when(F.upper(F.col('grade')).isin('D'), 'D')
     .when(F.upper(F.col('grade')).isin('F'), 'F')
     .otherwise('C'))
df.select('grade').distinct().orderBy('grade').show()

### Step 5 - Filling NULL assignment_score with 0

In [0]:
print('NULL assignment_score:', df.filter(F.col('assignment_score').isNull()).count())
df = df.fillna(0.0, subset=['assignment_score'])

### Step 6 - Type conversions

In [0]:
df = df.withColumn('dt', F.to_date(F.col('dt'),'yyyy-MM-dd'))\
       .withColumn('max_marks', F.col('max_marks').cast(IntegerType()))\
       .withColumn('assignment_score', F.col('assignment_score').cast(FloatType()))
df.printSchema()

### Step 7 - Final preview

In [0]:
display(df.limit(10))

In [0]:
df.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable('edutrack.silver.slv_enrollments')
print('slv_enrollments:', spark.table('edutrack.silver.slv_enrollments').count(),'rows')